# Spatial Integration

Loads the official administrative boundaries, validates the province/canton codes of each event, builds the event point layer and cross-checks each event's canton by point-in-polygon and by parish.

Outputs: `hazard_event_geo.gpkg`, `admin_boundaries.gpkg` and simplified GeoJSON layers in `data/processed/`.

## 0. Setup

In [ ]:

# Standard library: system path handling
import sys
# Object-oriented file paths
from pathlib import Path

# Project root is the parent of the notebooks folder
PROJECT_ROOT = Path.cwd().parent
# Make the project's src package importable
sys.path.insert(0, str(PROJECT_ROOT))

# Helpers to load the YAML config and resolve folder paths
from src.utilidades.config_utils import load_config, resolve_path
# Project logger factory
from src.utilidades.logging_utils import get_logger
# Helper to create output folders
from src.utilidades.io_utils import ensure_dir

# Tabular data handling
import pandas as pd
# Geospatial data handling
import geopandas as gpd
# Point geometry for event locations
from shapely.geometry import Point

# Show up to 30 columns when printing DataFrames
pd.set_option("display.max_columns", 30)

In [ ]:
# Path to the central configuration file
CONFIG_PATH = PROJECT_ROOT / "config" / "config.yaml"
# Load the configuration into a dictionary
config = load_config(CONFIG_PATH)

# Create a logger that writes to the logs folder
logger = get_logger("step2_spatial_integration", config, log_dir=PROJECT_ROOT / "logs")
# Log that the notebook started and which config was used
logger.info("integración espacial iniciada (v2, con confirmación por parroquia) — configuración cargada desde %s", CONFIG_PATH)

# Coordinate reference system used across the project
PROJECT_CRS = config["project"]["crs"]

## 1. Load and standardize administrative boundaries

In [ ]:
# Folder with the raw administrative boundary files
admin_dir = resolve_path(config, "raw_dir", base_dir=Path.cwd()) / "admin_boundaries"
# File names of each administrative level, from the config
admin_files = config["admin_boundaries"]["files"]
# Prefix to strip from official place codes
pcode_prefix = config["admin_boundaries"]["pcode_prefix"]

# Log where the boundaries are loaded from
logger.info("Cargando límites administrativos desde %s", admin_dir)

# Read the country boundary
adm0 = gpd.read_file(admin_dir / admin_files["adm0"], engine="pyogrio")
# Read the province boundaries
adm1 = gpd.read_file(admin_dir / admin_files["adm1"], engine="pyogrio")
# Read the canton boundaries
adm2 = gpd.read_file(admin_dir / admin_files["adm2"], engine="pyogrio")
# Read the parish boundaries
adm3 = gpd.read_file(admin_dir / admin_files["adm3"], engine="pyogrio")

# Loop over all levels to report their size
for name, gdf in [("adm0", adm0), ("adm1", adm1), ("adm2", adm2), ("adm3", adm3)]:
    # Log the number of features and the CRS of each level
    logger.info("%s: %d features, CRS=%s", name, len(gdf), gdf.crs)

In [ ]:
# Reproject the country layer to the project CRS
adm0 = adm0.to_crs(PROJECT_CRS)
# Reproject the province layer
adm1 = adm1.to_crs(PROJECT_CRS)
# Reproject the canton layer
adm2 = adm2.to_crs(PROJECT_CRS)
# Reproject the parish layer
adm3 = adm3.to_crs(PROJECT_CRS)

# Convert an official place code to the national DPA code
def pcode_to_dpa(series: pd.Series, prefix: str) -> pd.Series:
    """Convert an official place code (for example 'EC09') to the national DPA code (for example '09').
    
    Args:
        series: Place codes with the country prefix.
        prefix: Country prefix to remove.
    
    Returns:
        The codes without the prefix.
    """
    # Remove the country prefix from the code
    return series.str.replace(prefix, "", regex=False)

# Province code in DPA format
adm1["province_code"] = pcode_to_dpa(adm1["ADM1_PCODE"], pcode_prefix)
# Canton code in DPA format
adm2["canton_code"] = pcode_to_dpa(adm2["ADM2_PCODE"], pcode_prefix)
# Parish code in DPA format
adm3["parish_code"] = pcode_to_dpa(adm3["ADM3_PCODE"], pcode_prefix)

# Standardize province column names
adm1 = adm1.rename(columns={"ADM1_ES": "province_name_official"})
# Standardize canton column names
adm2 = adm2.rename(columns={"ADM2_ES": "canton_name_official", "ADM1_ES": "province_name_official"})
# Standardize parish column names
adm3 = adm3.rename(columns={"ADM3_ES": "parish_name_official", "ADM2_ES": "canton_name_official"})

# Canton code that each parish belongs to
adm3["canton_code_from_parish"] = pcode_to_dpa(adm3["ADM2_PCODE"], pcode_prefix)

# Preview the first three cantons
adm2[["canton_code", "canton_name_official", "province_name_official", "geometry"]].head(3)

## 2. Validate the code match between boundaries and events

In [ ]:
# Folder with processed tables
processed_dir = resolve_path(config, "processed_dir", base_dir=Path.cwd())
# Load the hazard events produced by the ingestion notebook
hazard_event = pd.read_parquet(processed_dir / "hazard_event.parquet")
# Log the number of events loaded
logger.info("hazard_event cargado: %d filas", len(hazard_event))

# Zero-pad province codes to 2 digits
hazard_event["province_code"] = hazard_event["province_code"].astype(str).str.zfill(2)
# Zero-pad canton codes to 4 digits
hazard_event["canton_code"] = hazard_event["canton_code"].astype(str).str.zfill(4)
# Zero-pad parish codes to 6 digits
hazard_event["parish_code"] = hazard_event["parish_code"].astype(str).str.zfill(6)

# Provinces present in the events
he_provinces = set(hazard_event["province_code"].unique())
# Provinces present in the official boundaries
adm1_provinces = set(adm1["province_code"].unique())
# Cantons present in the events
he_cantons = set(hazard_event["canton_code"].unique())
# Cantons present in the official boundaries
adm2_cantons = set(adm2["canton_code"].unique())

# Log the province coverage
logger.info(
    "Provincias — en hazard_event sin match en ADM1: %d | en ADM1 sin match en hazard_event: %d",
    len(he_provinces - adm1_provinces), len(adm1_provinces - he_provinces),
)
# Log the canton coverage
logger.info(
    "Cantones — en hazard_event sin match en ADM2: %d | en ADM2 sin match en hazard_event: %d",
    len(he_cantons - adm2_cantons), len(adm2_cantons - he_cantons),
)

# Check every event province exists in the boundaries
assert he_provinces <= adm1_provinces, "Hay provincias en hazard_event sin límite administrativo"
# Check every event canton exists in the boundaries
assert he_cantons <= adm2_cantons, "Hay cantones en hazard_event sin límite administrativo"
# Log that the code match is complete
logger.info("Validación de cruce de códigos: OK (100%% de cobertura provincia y cantón)")

## 3. Build the event point layer

In [ ]:
# Build a point for every event from its longitude and latitude
geometry = [Point(xy) for xy in zip(hazard_event["longitude"], hazard_event["latitude"])]

# Create the GeoDataFrame of events
hazard_event_geo = gpd.GeoDataFrame(hazard_event, geometry=geometry, crs=PROJECT_CRS)

# Log the number of points
logger.info("hazard_event_geo construido: %d puntos", len(hazard_event_geo))
# Preview the first two rows
hazard_event_geo.head(2)

## 4. Point-in-polygon validation

In [ ]:
# Keep only the canton columns needed for the join
adm2_for_join = adm2[["canton_code", "canton_name_official", "province_name_official", "geometry"]]
# Spatial join: find the canton polygon that contains each point
joined = gpd.sjoin(
    hazard_event_geo,
    adm2_for_join.rename(columns={"canton_code": "canton_code_geo", "canton_name_official": "canton_name_geo"}),
    how="left",
    predicate="within",
)

# Row count before removing duplicates
n_before = len(joined)
# Remove duplicates caused by points on shared borders
joined = joined[~joined.index.duplicated(keep="first")]
# Log the number of duplicated rows removed
logger.info("Filas duplicadas por sjoin en bordes compartidos: %d", n_before - len(joined))

# Flag events whose reported canton equals the geometric canton
joined["canton_geo_matches_reported"] = joined["canton_code_geo"] == joined["canton_code"]

# Share of events where both agree
match_rate = joined["canton_geo_matches_reported"].mean()
# Count events that fall in no polygon
n_no_polygon = joined["canton_code_geo"].isna().sum()
# Log the agreement rate and the unmatched count
logger.info(
    "Validación espacial: %.1f%% de coincidencia cantón reportado vs. geométrico | %d puntos sin polígono",
    match_rate * 100, n_no_polygon,
)

In [ ]:
# Events where reported and geometric cantons differ
mismatches = joined.loc[~joined["canton_geo_matches_reported"] & joined["canton_code_geo"].notna()]
# Log the number of disagreements
logger.info("Eventos con cantón reportado distinto del geométrico: %d de %d", len(mismatches), len(joined))
# Show the provinces with most disagreements
mismatches["province"].value_counts().head(10)

## 4b. Parish cross-check

In [ ]:
# Map from each parish to its canton
parish_to_canton = dict(zip(adm3["parish_code"], adm3["canton_code_from_parish"]))

# Look up the canton implied by the reported parish
joined["canton_code_from_parish"] = joined["parish_code"].map(parish_to_canton)

# Flag events whose reported canton agrees with the parish
joined["canton_confirmed_by_parish"] = joined["canton_code_from_parish"] == joined["canton_code"]

# Number of events with parish information
n_with_parish = joined["canton_code_from_parish"].notna().sum()
# Parish coverage as a percentage
pct_parish_coverage = n_with_parish / len(joined) * 100
# Log the parish coverage
logger.info(
    "Cobertura de parish_code contra ADM3 2024: %.1f%% (%d de %d eventos)",
    pct_parish_coverage, n_with_parish, len(joined),
)

# Share of events confirmed by the parish
confirm_rate = joined.loc[joined["canton_code_from_parish"].notna(), "canton_confirmed_by_parish"].mean()
# Log the confirmation rate
logger.info(
    "De los eventos con parroquia válida, confirman el cantón reportado por SNGRE: %.1f%%",
    confirm_rate * 100,
)

In [ ]:
# Events outside every polygon
sin_poligono = joined[joined["canton_code_geo"].isna()]
# How many of those the parish confirms
resueltos_sin_poligono = sin_poligono["canton_confirmed_by_parish"].sum()
# Log how many were resolved
logger.info(
    "De los %d eventos sin polígono cantonal: %d quedan confirmados por parroquia (cantón reportado correcto)",
    len(sin_poligono), resueltos_sin_poligono,
)

# Events whose geometry disagrees with the report
geo_distinto = joined.loc[~joined["canton_geo_matches_reported"] & joined["canton_code_geo"].notna()]
# How many of those the parish confirms
resueltos_geo_distinto = geo_distinto["canton_confirmed_by_parish"].sum()
# Log how many were resolved
logger.info(
    "De los %d eventos con cantón geométrico distinto del reportado: %d quedan confirmados por parroquia",
    len(geo_distinto), resueltos_geo_distinto,
)

# Assign a confidence label to an event's canton
def confidence_label(row) -> str:
    """Assign a confidence label to the canton of one event using the geometric and the parish checks.
    
    Args:
        row: Event row with the check columns.
    
    Returns:
        'alta' when either check confirms the canton, 'baja_sin_parroquia' when neither source is available, otherwise 'revisar'.
    """
    # Geometry agrees with the report
    geo_ok = row.get("canton_geo_matches_reported") is True
    # Parish agrees with the report
    parish_ok = row.get("canton_confirmed_by_parish") is True
    # Parish information is available
    parish_available = pd.notna(row.get("canton_code_from_parish"))
    # Either source confirms the canton
    if geo_ok or parish_ok:
        # Label for confirmed events
        return "alta"
    # No parish and no polygon: low confidence
    if not parish_available and pd.isna(row.get("canton_code_geo")):
        # Label for events with no second source
        return "baja_sin_parroquia"
    # Otherwise the event needs manual review
    return "revisar"

# Apply the label to every event
joined["canton_confidence"] = joined.apply(confidence_label, axis=1)
# Log the distribution of confidence labels
logger.info("Distribución de confianza final por evento: %s", joined["canton_confidence"].value_counts().to_dict())
# Show the distribution of confidence labels
joined["canton_confidence"].value_counts()

## 5. Harmonize official names

In [ ]:
# Drop the helper column left by the spatial join
hazard_event_geo_final = joined.drop(columns=["index_right"], errors="ignore")

# Count name variants per official province
name_variants = hazard_event_geo_final.groupby("province_name_official")["province"].nunique()
# Log how many provinces have several spellings in the source
logger.info("Provincias con más de una variante de nombre en SNGRE: %d de %d", (name_variants > 1).sum(), len(name_variants))

## 6. Data-quality summary

In [ ]:
# Dictionary with the quality summary
summary = {
    "total_eventos_geolocalizados": len(hazard_event_geo_final),
    "match_cantonal_geometrico_pct": round(match_rate * 100, 2),
    "eventos_sin_poligono_pct": round(n_no_polygon / len(joined) * 100, 2),
    "cobertura_parish_code_pct": round(pct_parish_coverage, 2),
    "confianza_alta_pct": round((hazard_event_geo_final["canton_confidence"] == "alta").mean() * 100, 2),
    "confianza_revisar_n": int((hazard_event_geo_final["canton_confidence"] == "revisar").sum()),
    "provincias_con_nombres_inconsistentes": int((name_variants > 1).sum()),
    "cantones_cubiertos_adm2": len(adm2),
    "parroquias_cubiertas_adm3": len(adm3),
}
# Print each summary item
for k, v in summary.items():
    # Print the key and its value
    print(f"{k}: {v}")
# Log the summary
logger.info("Resumen integración espacial: %s", summary)

## 7. Export

In [ ]:
# Create the output folder if needed
out_dir = ensure_dir(processed_dir)

# Temporary folder utilities
import tempfile
# File copy utility
import shutil

# Write to a temporary folder first to avoid partial files
with tempfile.TemporaryDirectory() as tmp:
    # Path of the temporary folder
    tmp_dir = Path(tmp)

    # Path of the temporary GeoPackage
    gpkg_tmp = tmp_dir / config["admin_boundaries"]["output_gpkg"]
    # Write the country layer
    adm0.to_file(gpkg_tmp, layer="adm0_country", driver="GPKG")
    # Write the province layer
    adm1.to_file(gpkg_tmp, layer="adm1_province", driver="GPKG")
    # Write the canton layer
    adm2.to_file(gpkg_tmp, layer="adm2_canton", driver="GPKG")
    # Write the parish layer
    adm3.to_file(gpkg_tmp, layer="adm3_parish", driver="GPKG")

    # Simplification tolerance for web-friendly boundaries, in degrees
    SIMPLIFY_TOLERANCE_DEG = 0.0005
    # Copy the province layer
    adm1_simplified = adm1.copy()
    # Simplify province geometries
    adm1_simplified["geometry"] = adm1_simplified.geometry.simplify(SIMPLIFY_TOLERANCE_DEG, preserve_topology=True)
    # Copy the canton layer
    adm2_simplified = adm2.copy()
    # Simplify canton geometries
    adm2_simplified["geometry"] = adm2_simplified.geometry.simplify(SIMPLIFY_TOLERANCE_DEG, preserve_topology=True)

    # Write simplified provinces as GeoJSON
    adm1_simplified.to_file(tmp_dir / "adm1_province.geojson", driver="GeoJSON")
    # Write simplified cantons as GeoJSON
    adm2_simplified.to_file(tmp_dir / "adm2_canton.geojson", driver="GeoJSON")

    # Write the event points as a GeoPackage layer
    hazard_event_geo_final.to_file(tmp_dir / "hazard_event_geo.gpkg", layer="hazard_event", driver="GPKG")

    # List of files written to the final folder
    written_files = []
    # Loop over the temporary files
    for f in tmp_dir.iterdir():
        # Destination path in the processed folder
        dest = out_dir / f.name
        # Copy the file to its destination
        shutil.copy2(f, dest)
        # Remember the destination
        written_files.append(dest)
        # Log the file name and size
        logger.info("Copiado a destino final: %s (%.1f MB)", dest.name, dest.stat().st_size / 1e6)

# Print the output folder
print("Archivos escritos en:", out_dir)
# Loop over the written files
for f in written_files:
    # Print each file name
    print(" -", f.name)